# JI-ADF research baseline v1 — MILK10k

Implementation based on [Nguyen et al., JI-ADF, arXiv:2604.27343v2](https://arxiv.org/html/2604.27343v2): modality-specific attention + residual path, image/metadata/joint heads, and adaptive probability mixing with auxiliary supervision.

**The next research baseline is `ji-adf-v1`. It is not trained yet.** This notebook trains and freezes it after recording results. The historical v1 (macro F1 **0.402574**, accuracy **0.635496**) and your completed Kaggle notebook stay unchanged.

Create a new notebook. Attach **MILK10k Training** and the new **Skin Cancer JI ADF** code ZIP dataset. Select **T4 x2**. ImageNet weights, old v1 weights/validator, exact split CSV and source are included. No model download on Kaggle.

Default: one JI-ADF job and a matched simpler control, same seed, one per T4. JI-ADF is the predeclared reference even if the control scores better. This is a documented T4 adaptation, not a claim to reproduce the paper's test score.

In [ ]:
from pathlib import Path
import os, sys, subprocess, shutil, json, zipfile, stat

CODE_INPUT = None  # e.g. Path('/kaggle/input/datasets/aachintya/skin-cancer-ji-adf')
DATA_INPUT = None  # e.g. Path('/kaggle/input/datasets/aachintya/milk10k-training')
RESUME_INPUT = None  # Previous ji_adf_artifacts.zip, NOT the small frozen baseline ZIP.

PROJECT = Path('/kaggle/working/skin-cancer-ji-adf')
RUN = PROJECT / 'experiments/ji-adf-v1-fp32'
VARIANTS = ['ji_adf', 'concat_fixed']  # Primary + matched control; parallel on two T4s.
SEEDS = [42]                         # Add seeds before starting for an ensemble recipe.
INITIALIZATION = 'imagenet'           # Bundled weights; optional 'v1' is a different experiment.
EPOCHS = 50
IMAGE_SIZE = 384
BATCH_SIZE = 4
ACCUMULATE = 4                       # Effective batch 16. If OOM before training: batch 2, accumulate 8.
NUM_WORKERS = 0                      # Reliable default. Optional >0 uses spawn and timeout.
BUILD_RGB_CACHE = True
CACHE_THREADS = 4
REPRODUCE_BASELINE = True             # Archived old source + paired uncertainty.
FROZEN = PROJECT / 'saved_references/ji-adf-v1'
ARTIFACT_ZIP = Path('/kaggle/working/ji_adf_fp32_artifacts.zip')
BASELINE_ZIP = Path('/kaggle/working/ji_adf_baseline_v1.zip')
print('JI-ADF configuration ready. Historical baseline output paths are never used.', flush=True)

## Environment preflight

Keep Kaggle's installed CUDA torch/torchvision. The package includes ImageNet initialization and the historical model, so internet is needed only if a missing/incompatible dependency must be installed. The old sklearn fusion checkpoint requires sklearn 1.6.1.

In [ ]:
print('Python is responding; importing PyTorch...', flush=True)
import torch, torchvision, importlib.util
if not torch.cuda.is_available():
    raise RuntimeError('CUDA unavailable: enable GPU T4 x2 in notebook settings, then restart and Run All.')
print('PyTorch:', torch.__version__, 'torchvision:', torchvision.__version__, flush=True)
for i in range(torch.cuda.device_count()):
    print(f'GPU {i}: {torch.cuda.get_device_name(i)}', flush=True)
required = {'numpy': 'numpy', 'pandas': 'pandas', 'scipy': 'scipy', 'PIL': 'Pillow',
            'cv2': 'opencv-python-headless', 'joblib': 'joblib', 'sklearn': 'scikit-learn==1.6.1'}
missing = [pkg for module, pkg in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', *missing], check=True, timeout=600)
from importlib.metadata import version
if version('scikit-learn') != '1.6.1':
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'scikit-learn==1.6.1'], check=True, timeout=600)
    if 'sklearn' in sys.modules:
        raise RuntimeError('sklearn was already imported. Restart the kernel once, then Run All.')
print('Environment ready. Bundled ImageNet and historical weights are ready for offline use.', flush=True)

## Locate the new JI-ADF package and existing MILK10k images

Discovery looks only a few levels deep. If multiple matching inputs are found, enter their exact paths in the configuration cell. This does not recursively scan thousands of lesion directories.

Setup restores the bundled opaque source archive to a separate writable reference and verifies every original SHA-256. If using the first upload, attach the small `ji_adf_reference_fix.zip` repair input too.


In [ ]:
INPUT = Path('/kaggle/input')
roots = {p for p in INPUT.iterdir() if p.is_dir()}
roots.update(p for p in INPUT.glob('datasets/*/*') if p.is_dir())
shallow = set(roots)
for root in roots:
    shallow.update(p for p in root.iterdir() if p.is_dir())

def unique(paths, label):
    paths = list({p.resolve(): p for p in paths}.values())
    if len(paths) != 1:
        raise ValueError(f'Expected one {label}, found {paths}. Set its exact input path in the configuration cell.')
    return paths[0]

if CODE_INPUT is None:
    code_dirs = [p for p in shallow if (p / 'src/ji_adf/model.py').is_file() and (p / 'reference/manifest.json').is_file()]
    code_zips = [p / 'skin_cancer_ji_adf.zip' for p in shallow if (p / 'skin_cancer_ji_adf.zip').is_file()]
    CODE_INPUT = unique(code_dirs + code_zips, 'JI-ADF code dataset')
CODE_INPUT = Path(CODE_INPUT)
PROJECT.mkdir(parents=True, exist_ok=True)

def safe_extract(source, destination):
    destination = Path(destination).resolve()
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(source) as archive:
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            if not target.is_relative_to(destination) or stat.S_ISLNK(member.external_attr >> 16):
                raise ValueError(f'Unsafe archive member: {member.filename}')
        archive.extractall(destination)

if CODE_INPUT.is_file():
    safe_extract(CODE_INPUT, PROJECT)
else:
    for name in ['src', 'tests', 'notebooks']:
        shutil.copytree(CODE_INPUT / name, PROJECT / name, dirs_exist_ok=True)
    reference_target = PROJECT / 'reference'
    if reference_target.is_symlink() and reference_target.resolve() != (CODE_INPUT / 'reference').resolve():
        raise ValueError('Project reference points to another upload. Use a new PROJECT directory.')
    if not reference_target.exists():
        reference_target.symlink_to((CODE_INPUT / 'reference').resolve(), target_is_directory=True)

if DATA_INPUT is None:
    DATA_INPUT = unique([p for p in shallow if (p / 'MILK10k_Training_Input').is_dir()
                         and (p / 'MILK10k_Training_Metadata.csv').is_file()], 'MILK10k data dataset')
DATA_INPUT = Path(DATA_INPUT)
IMAGES = DATA_INPUT / 'MILK10k_Training_Input'
REFERENCE = PROJECT / 'reference'
SRC = PROJECT / 'src'
sys.path.insert(0, str(SRC))
# Kaggle may unpack nested ZIPs. Restore exact source-archive bytes locally.
# The source dataset and its checksum manifest remain unchanged.
import runpy
preparer = SRC / 'scripts/prepare_ji_adf_reference.py'
snapshot = None
if not preparer.is_file() or (not (REFERENCE / 'source_snapshot.zip').is_file()
                             and not (REFERENCE / 'source_snapshot.zip.bin').is_file()):
    repair_scripts = [*INPUT.glob('*/repair_ji_adf_reference.py'),
                      *INPUT.glob('datasets/*/*/repair_ji_adf_reference.py')]
    if not repair_scripts:
        raise FileNotFoundError('Attach ji_adf_reference_fix.zip as an additional input dataset, '
                                'or attach the corrected skin_cancer_ji_adf.zip upload.')
    preparer = unique(repair_scripts, 'JI-ADF reference repair dataset')
    snapshot = preparer.with_name('source_snapshot.zip.bin')
print('Verifying the reference and preparing a writable copy...', flush=True)
REFERENCE = runpy.run_path(str(preparer))['prepare_reference'](
    REFERENCE, PROJECT / 'reference_verified', snapshot=snapshot)
from ji_adf.runtime import verify_reference
reference_manifest = verify_reference(REFERENCE)
from utils.baselines import sha256_file
for name, digest in reference_manifest['raw_release_sha256'].items():
    if name.endswith('.csv') and (DATA_INPUT / name).is_file():
        if sha256_file(DATA_INPUT / name) != digest:
            raise ValueError(f'Raw dataset differs from the baseline release: {name}')
from utils.kaggle import verify_split
verify_split(REFERENCE / 'split.csv')
print('Code:', CODE_INPUT, '\nImages:', IMAGES, '\nOutput:', RUN, flush=True)

## Optional resume and real GPU model check

Resume only with the large **ji_adf_fp32_artifacts.zip**. Keep all settings/code unchanged. Checkpoints restore model, optimizer, scheduler, scaler and RNG state at the last completed epoch.

The smoke test runs three actual optimizer steps, including gradient accumulation, for each planned architecture before the cache or long run. If memory is insufficient, lower the microbatch and increase accumulation before starting a fresh run.

Training and inference use FP32. A T4 diagnostic reproduced nonfinite FP16 encoder activations with finite inputs and weights; the same weights produced finite FP32 outputs. The cache is reused, and the fixed run uses a separate output folder.


In [ ]:
if RESUME_INPUT is not None:
    previous = Path(RESUME_INPUT)
    if list(RUN.glob('seed-*/resume.pt')):
        print('Current local resume states already exist; keeping them.', flush=True)
    elif previous.is_file():
        safe_extract(previous, RUN)
    else:
        shutil.copytree(previous, RUN, dirs_exist_ok=True)
    print('Resume plan:', (RUN / 'run_plan.json').read_text(), flush=True)

def run_script(name, *args):
    command = [sys.executable, '-u', str(SRC / 'scripts' / name), *map(str, args)]
    import signal
    process = subprocess.Popen(command, cwd=PROJECT, start_new_session=True)
    try:
        code = process.wait()
        if code:
            raise subprocess.CalledProcessError(code, command)
    except BaseException:
        if process.poll() is None:
            # The training orchestrator handles SIGTERM and stops its own GPU children.
            os.killpg(process.pid, signal.SIGTERM)
            try:
                process.wait(timeout=30)
            except subprocess.TimeoutExpired:
                os.killpg(process.pid, signal.SIGKILL)
                process.wait()
        raise

print('Checking JI-ADF and control GPU memory/gradients...', flush=True)
for index, variant in enumerate(VARIANTS):
    gpu = index % torch.cuda.device_count()
    run_script('ji_adf.py', 'smoke', '--reference', REFERENCE, '--device', f'cuda:{gpu}',
               '--variant', variant, '--initialization', INITIALIZATION,
               '--batch-size', BATCH_SIZE, '--size', IMAGE_SIZE, '--accumulate', ACCUMULATE)
print('GPU model smoke tests passed. Preparing data.', flush=True)

## Fast lossless RGB cache

Full RGB image context, aspect ratio preserved with neutral padding. No expensive hair inpainting or CLAHE. Threads avoid the old fork deadlock; progress prints every 500 images. Existing files are reused.

In [ ]:
from improved.data import build_rgb_cache, paired_frame, original_path
RGB_CACHE = PROJECT / 'rgb_cache'
if BUILD_RGB_CACHE:
    build_rgb_cache(REFERENCE / 'split.csv', IMAGES, RGB_CACHE,
                    side=max(384, IMAGE_SIZE), workers=CACHE_THREADS)
else:
    RGB_CACHE = None
    pairs = paired_frame(REFERENCE / 'split.csv')
    for row in pairs.itertuples():
        original_path(IMAGES, row.lesion_id, row.clinical_id)
        original_path(IMAGES, row.lesion_id, row.dermoscopic_id)
print('Data ready. The next cell starts actual disease-model training.', flush=True)

## Train the JI-ADF reference and matched control

The full JI-ADF and simpler `concat_fixed` control start from identical ImageNet weights with the same seed and training recipe. They use the same auxiliary loss; the control removes both attention and adaptive decision mixing.

Additional optional controls are `mmfa_fixed` (attention, equal branch weights) and `concat_adaptive` (concatenation, adaptive weights). Set variants before starting a new experiment. One GPU runs jobs sequentially; two T4s run two jobs concurrently.

Watch `EPOCH ... DONE` for validation macro F1 and measured epoch duration. Up to 50 epochs, early stopping after 10 non-improving epochs. Cancellation cleans up both GPU jobs and exports completed epochs.

In [ ]:
from ji_adf.runtime import export_run
command = ['run', '--reference', REFERENCE, '--images', IMAGES, '--output', RUN,
           '--variants', *VARIANTS, '--seeds', *SEEDS, '--initialization', INITIALIZATION,
           '--epochs', EPOCHS, '--batch-size', BATCH_SIZE, '--accumulate', ACCUMULATE,
           '--size', IMAGE_SIZE, '--workers', NUM_WORKERS]
if RGB_CACHE:
    command += ['--cache', RGB_CACHE]
try:
    run_script('ji_adf.py', *command)
finally:
    if RUN.exists():
        export_run(RUN, ARTIFACT_ZIP)
        print('Saved resumable output:', ARTIFACT_ZIP, flush=True)

## Lock model identities, evaluate, and compare with historical v1

Checkpoint selection uses validation macro F1 only. There is no prior/threshold tuning. Primary JI-ADF and seed weights are fixed before holdout; fusion results are diagnostic. All four split assignments stay exactly as saved v1.

The table compares the same 524 held-out lesions, argmax, no TTA. The original v1 is reproduced using its archived source and weights before paired bootstrap intervals are computed. Its old preprocessing can take several minutes. The upload gate is bypassed only on this curated benchmark; actual upload inference requires it on both originals.

In [ ]:
command = ['evaluate', '--reference', REFERENCE, '--images', IMAGES, '--output', RUN,
           '--device', 'cuda:0', '--batch-size', BATCH_SIZE]
if RGB_CACHE:
    command += ['--cache', RGB_CACHE]
if REPRODUCE_BASELINE:
    command += ['--reproduce-baseline']
try:
    run_script('ji_adf.py', *command)
finally:
    export_run(RUN, ARTIFACT_ZIP)
print((RUN / 'comparison.md').read_text(), flush=True)
import pandas as pd
from IPython.display import display
report = json.loads((RUN / 'metrics/ji_adf.holdout.json').read_text())
display(pd.DataFrame(report['per_class']).T)

## Preserve the completed run as our new research baseline

Freeze **ji-adf-v1** as a separate read-only inference snapshot with metrics, exact splits, source and checksums. This preserves the measured JI-ADF result even if it does not outperform historical v1. It does not deploy into the local app. Existing snapshots are verified and never overwritten.

Future improvements can compare with this snapshot using `baseline.py compare --baseline /path/to/ji-adf-v1 --candidate /path/to/future.metrics.json`. The historical baseline remains available for comparison too.


In [ ]:
if FROZEN.exists():
    run_script('ji_adf.py', 'verify', FROZEN)
    if json.loads((FROZEN / 'inference.json').read_text()) != json.loads((RUN / 'inference.json').read_text()):
        raise ValueError('Frozen JI-ADF reference belongs to another run. Choose a new FROZEN destination/name.')
else:
    run_script('ji_adf.py', 'freeze', '--run', RUN, '--destination', FROZEN, '--name', 'ji-adf-v1')
export_run(FROZEN, BASELINE_ZIP)
print('New trained research baseline preserved:', BASELINE_ZIP, flush=True)


## Download both outputs

- **ji_adf_fp32_artifacts.zip**: all jobs, optimizer resume states, histories, source, metrics and predictions.
- **ji_adf_baseline_v1.zip**: the small immutable research reference, including best JI-ADF weights and validator; optimizer tensors excluded.

Inspect macro F1, balanced accuracy, per-class recall and uncertainty. Higher accuracy alone does not establish better rare-class recognition. This internal split has tiny rare-class support and is already known; use an untouched external evaluation for stronger claims.

Do not feed these new-format ZIPs to the current v1 Streamlit importer. The new `ji_adf.predictor.JIADFPredictor` validates both original uploads before disease inference.

In [ ]:
from IPython.display import FileLink, display
from ji_adf.runtime import export_run
export_run(RUN, ARTIFACT_ZIP)
display(FileLink(str(ARTIFACT_ZIP)))
if FROZEN.exists():
    export_run(FROZEN, BASELINE_ZIP)
    display(FileLink(str(BASELINE_ZIP)))
for name in ['comparison.md', 'comparison.json', 'metrics/ji_adf.holdout.json', 'predictions/ji_adf.holdout.csv']:
    if (RUN / name).is_file():
        display(FileLink(str(RUN / name)))